# WhisperX PT-BR Transcription and Diarization

Este notebook demonstra como usar a biblioteca **whisperX** para transcrever e realizar diarização de um vídeo em português do Brasil.

In [ ]:
!pip install -q whisperx

In [ ]:
import whisperx
import torch

# Defina o caminho do arquivo de vídeo ou áudio
audio_file = 'caminho/para/video.mp4'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'float32'

# 1. Transcrição com whisper
model = whisperx.load_model('large-v2', device=device, compute_type=compute_type, language='pt')

audio = whisperx.load_audio(audio_file)
result = model.transcribe(audio, batch_size=16)

# 2. Alinhamento com wav2vec2
model_a, metadata = whisperx.load_align_model(language_code='pt', device=device)
result = whisperx.align(result['segments'], model_a, metadata, audio, device)

# 3. Diarização (necessita token da HF para pyannote)
hf_token = 'YOUR_HF_TOKEN'
diarize_model = whisperx.diarize.DiarizationPipeline(use_auth_token=hf_token, device=device)
diarize_segments = diarize_model(audio)

result = whisperx.assign_word_speakers(diarize_segments, result)

# Exibir os primeiros segmentos com oradores
for seg in result['segments'][:5]:
    print(f"{seg['speaker']}: {seg['text']}")

Execute todas as células acima para obter a transcrição com os identificadores de falantes.